## Building our Own Chatbot using Memory in LangChain:

Cell 1 - Imports and setup:<br>
We'll try using all three types of Memories to understand their logic:<br>
ChatMessageHistory<br>
ConversationBufferMemory<br>
ConversationSummaryMemory<br>

In [1]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory
)
from langchain.chains import ConversationChain
from langchain.memory import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    api_key=os.getenv("GROQ_API_KEY")
)

Cell 2 - Initial history setup:

In [7]:
# Pre-load some context before the conversation starts
history = ChatMessageHistory()
history = ChatMessageHistory()
history.add_user_message("Hello, my name is Alice.")
history.add_ai_message("Hi Alice! How are you? Do you have any questions for me?")
history.add_user_message("I'm Fine. Do you know when is the best time to apply for a Driving License?")
history.add_ai_message("The best time to apply for a driving license is typically when you're 16-18 years old depending on your country. You should apply at least 2-3 months before you need it to account for processing time and test scheduling.")

# Print initial history
print("Initial Chat History:")
for message in history.messages:
    sender = "Human" if isinstance(message, HumanMessage) else "AI"
    print(f"{sender}: {message.content}")

Initial Chat History:
Human: Hello, my name is Alice.
AI: Hi Alice! How are you? Do you have any questions for me?
Human: I'm Fine. Do you know when is the best time to apply for a Driving License?
AI: The best time to apply for a driving license is typically when you're 16-18 years old depending on your country. You should apply at least 2-3 months before you need it to account for processing time and test scheduling.


Cell 3 - Chat simulation function:

In [ ]:
# A chat_simulation is a way to invoke inputs rather than using:
# conversation.invoke(input=" ") everytime..
def chat_simulation(conversation, inputs, label=""):
    print(f"\n=== {label} ===")
    for i, user_input in enumerate(inputs):
        print(f"\n--- Turn {i+1} ---")
        print(f"Human: {user_input}")
        response = conversation.invoke(input=user_input)
        print(f"AI: {response['response']}")
    print(f"\n=== End of {label} ===")

test_inputs = [
    "What documents do I need for the driving license application?",
    "How much does it cost?",
    "What is the test like?",
    "What was my name again?",
    "Can you summarize what we discussed about driving licenses?"
]

Cell 4 - Memory Type 1: ConversationBufferMemory:

In [9]:
# Stores EVERYTHING — full conversation history
print("\n" + "="*50)
print("MEMORY TYPE 1: ConversationBufferMemory")
print("Stores the complete conversation history")
print("="*50)

buffer_memory = ConversationBufferMemory(chat_memory=history)
buffer_conversation = ConversationChain(
    llm=llm,
    memory=buffer_memory,
    verbose=False  # set True if you want to see full prompts
)

chat_simulation(buffer_conversation, test_inputs, "Buffer Memory Chat")

print("\nFinal Buffer Memory Contents:")
print(buffer_memory.buffer)
print(f"\nBuffer Memory Size: {len(buffer_memory.buffer)} characters")


MEMORY TYPE 1: ConversationBufferMemory
Stores the complete conversation history

=== Buffer Memory Chat ===

--- Turn 1 ---
Human: What documents do I need for the driving license application?
AI: For a driving license application, you'll typically need to provide several documents. These may vary depending on your location, but common requirements include a valid proof of identity, such as a passport or birth certificate, and proof of residency, like a utility bill or lease agreement. You may also need to provide a social security number or equivalent, and in some cases, proof of citizenship or immigration status. Additionally, you'll likely need to pass a vision test, and provide a photograph that meets the DMV's specifications. It's also a good idea to check with your local DMV for their specific requirements, as they can vary. For example, in the United States, the DMV in California may require different documents than the DMV in New York. Does that help, Alice?

--- Turn 2 ---
H

Inference:<br>
Due to having memory, the AI can associate "it" with the "Driving Licence".<br>
Also, it remembered "Alice" from the very beginning and answered "Does that help, Alice?" naturally in Turn 1. Perfect recall, but keeps growing.<br>

Cell 5 - Memory Type 2: ConversationBufferWindowMemory:

In [10]:
# Only remembers last K exchanges
print("\n" + "="*50)
print("MEMORY TYPE 2: ConversationBufferWindowMemory")
print("Only remembers last 2 exchanges (k=2)")
print("="*50)

window_memory = ConversationBufferWindowMemory(k=2)
# Pre-load initial context
# Each call saves one exchange (one human + one AI message)
window_memory.save_context(
    {"input": "Hello, my name is Alice."},
    {"output": "Hi Alice! How are you? Do you have any questions for me?"}
)

window_memory.save_context(
    {"input": "I'm Fine. Do you know when is the best time to apply for a Driving License?"},
    {"output": "The best time to apply for a driving license is typically when you're 16-18 years old depending on your country. You should apply at least 2-3 months before you need it to account for processing time and test scheduling."}
)

window_conversation = ConversationChain(
    llm=llm,
    memory=window_memory,
    verbose=False
)

chat_simulation(window_conversation, test_inputs, "Window Memory Chat")

print("\nFinal Window Memory Contents:")
print(window_memory.buffer)
print(f"\nWindow Memory Size: {len(window_memory.buffer)} characters")


MEMORY TYPE 2: ConversationBufferWindowMemory
Only remembers last 2 exchanges (k=2)

=== Window Memory Chat ===

--- Turn 1 ---
Human: What documents do I need for the driving license application?
AI: For a driving license application, you'll typically need to provide several documents. These may vary depending on your country or state, but I can give you a general idea of what's usually required. 

You'll likely need to provide proof of identity, such as a valid passport or a birth certificate. You may also need to show proof of residency, which could be a utility bill or a bank statement with your address on it. Additionally, you'll probably need to provide a social security number or a taxpayer identification number, depending on where you live.

In the United States, for example, you would typically need to provide one proof of identity, one proof of social security number, and two proofs of residency. Some states may also require a vision test, so be prepared for that as well.

I

Inference:<br>
This time, the AI only summarized the test, because with k=2 it only remembered the last 2 exchanges (Turn 3 about the test and Turn 4 about the name). It completely forgot about documents and costs from earlier turns.<br>
That's window memory's limitation - it forgets old context.<br>

Cell 6 - Memory Type 3: ConversationSummaryMemory:

In [11]:
# Summarizes conversation instead of storing full text
print("\n" + "="*50)
print("MEMORY TYPE 3: ConversationSummaryMemory")
print("Compresses conversation into a running summary")
print("="*50)

summary_memory = ConversationSummaryMemory(llm=llm)
# Each call saves one exchange (one human + one AI message)
summary_memory.save_context(
    {"input": "Hello, my name is Alice."},
    {"output": "Hi Alice! How are you? Do you have any questions for me?"}
)

summary_memory.save_context(
    {"input": "I'm Fine. Do you know when is the best time to apply for a Driving License?"},
    {"output": "The best time to apply for a driving license is typically when you're 16-18 years old depending on your country. You should apply at least 2-3 months before you need it to account for processing time and test scheduling."}
)

summary_conversation = ConversationChain(
    llm=llm,
    memory=summary_memory,
    verbose=False
)

chat_simulation(summary_conversation, test_inputs, "Summary Memory Chat")

print("\nFinal Summary Memory Contents:")
print(summary_memory.buffer)
print(f"\nSummary Memory Size: {len(summary_memory.buffer)} characters")


MEMORY TYPE 3: ConversationSummaryMemory
Compresses conversation into a running summary

=== Summary Memory Chat ===

--- Turn 1 ---
Human: What documents do I need for the driving license application?
AI: To apply for a driving license, you'll typically need to provide several documents. These may vary depending on your location, but I can give you a general idea of what's usually required. 

Firstly, you'll need to provide proof of identity, which can be a valid passport, a national ID card, or a birth certificate. Some countries may also accept other forms of identification, such as a state ID or a residency permit.

Next, you'll need to provide proof of residency, which can be a utility bill, a bank statement, or a lease agreement that shows your current address. This is usually required to confirm that you're a resident of the country or state where you're applying for the license.

You may also need to provide a social security number or a tax identification number, depending on

Inference:<br>
If we look at the Final Summary Contents, we notice that it captured everything in 2541 characters:<br>
-> Alice's introduction<br>
-> Driving license timing<br>
-> Documents needed<br>
-> Costs<br>
-> Test format<br>
-> Name recall<br>
-> Final summary<br>
All compressed from what would have been 6000+ characters. And it still answered correctly on every turn.<br>

Cell 7 - Comparing all three:

In [12]:
print("\n" + "="*50)
print("MEMORY COMPARISON")
print("="*50)
print(f"BufferMemory size:       {len(buffer_memory.buffer):>6} characters — stores everything")
print(f"WindowMemory size:       {len(window_memory.buffer):>6} characters — stores last {2} exchanges only")
print(f"SummaryMemory size:      {len(summary_memory.buffer):>6} characters — compressed summary")
print("\nKey insight:")
print("- Buffer: perfect recall, grows forever")
print("- Window: forgets old messages, stays small")
print("- Summary: compact but might lose details")


MEMORY COMPARISON
BufferMemory size:         6255 characters — stores everything
WindowMemory size:         1208 characters — stores last 2 exchanges only
SummaryMemory size:        2541 characters — compressed summary

Key insight:
- Buffer: perfect recall, grows forever
- Window: forgets old messages, stays small
- Summary: compact but might lose details
